# Deep Learning

# Tutorial 18: Hyperparameter Selection

In this tutorial, we will cover:

- What hyperparameters are and how they differ from trainable parameters
- Important hyperparameters in deep learning: learning rate, batch size, number of epochs, optimizer, regularization, architecture choices
- How hyperparameters influence convergence, stability, generalization, and training time
- Manual search, grid search, random search, and validation-based model selection
- Practical hyperparameter tuning with PyTorch
- Common failure modes such as overfitting, underfitting, exploding loss, and unstable training

Prerequisites:

- Python, PyTorch, Deep Learning Training, Stochastic Gradient Descent

My contact:

- Niklas Beuter (niklas.beuter@th-luebeck.de)

Course:

- Slides and notebooks will be available at https://lernraum.th-luebeck.de/course/view.php?id=5383

## Expected Outcomes
* Explain the difference between parameters and hyperparameters
* Identify important hyperparameters in neural network training
* Explain the effect of learning rate, batch size, epochs, optimizer, weight decay, and dropout
* Implement a small PyTorch training loop with configurable hyperparameters
* Compare multiple hyperparameter configurations using a validation set
* Interpret training and validation curves
* Select a suitable model configuration based on validation performance


## 1. Parameters vs. Hyperparameters

In deep learning, we distinguish between **parameters** and **hyperparameters**.

**Parameters** are learned from data during training. Examples are weights, biases, and convolution kernels.

**Hyperparameters** are chosen before or during training and are not directly learned by backpropagation. Examples are learning rate, batch size, number of epochs, optimizer type, architecture, dropout probability, and weight decay.


## 2. Setup

We use PyTorch and create a small synthetic classification dataset. The goal is not state-of-the-art performance, but to understand how hyperparameters affect training.


In [ ]:
import math
import random
from dataclasses import dataclass

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset, random_split
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE


## 3. Create a Synthetic Dataset

We create a two-class classification problem. The classes are not perfectly linearly separable, so a small neural network is useful.


In [ ]:
def make_moons_like_dataset(n_samples=1200, noise=0.18):
    """Create a simple two-class 2D dataset without relying on sklearn."""
    n_half = n_samples // 2
    theta1 = torch.rand(n_half) * math.pi
    x1 = torch.stack([torch.cos(theta1), torch.sin(theta1)], dim=1)
    theta2 = torch.rand(n_samples - n_half) * math.pi
    x2 = torch.stack([1.0 - torch.cos(theta2), 0.5 - torch.sin(theta2)], dim=1)
    X = torch.cat([x1, x2], dim=0)
    y = torch.cat([torch.zeros(n_half, dtype=torch.long), torch.ones(n_samples - n_half, dtype=torch.long)])
    X = X + noise * torch.randn_like(X)
    indices = torch.randperm(n_samples)
    return X[indices], y[indices]

X, y = make_moons_like_dataset()
plt.figure(figsize=(6, 5))
plt.scatter(X[:, 0], X[:, 1], c=y, s=12)
plt.title("Synthetic classification dataset")
plt.xlabel("x1")
plt.ylabel("x2")
plt.show()


Create a train/validation/test split with the following sizes:

- 70% training data
- 15% validation data
- 15% test data

Store the resulting datasets in the variables `train_dataset`, `val_dataset`, and `test_dataset`.

In [ ]:
full_dataset = TensorDataset(X, y)
n_total = len(full_dataset)
n_train = int(0.70 * n_total)
n_val = int(0.15 * n_total)
n_test = n_total - n_train - n_val
train_dataset, val_dataset, test_dataset = random_split(
    full_dataset,
    [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(SEED)
)
len(train_dataset), len(val_dataset), len(test_dataset)

## 4. Model Definition

Architecture is itself a hyperparameter. Here we define a small multilayer perceptron where the hidden size and dropout probability can be configured.


In [ ]:
class MLP(nn.Module):
    def __init__(self, input_dim=2, hidden_dim=32, output_dim=2, dropout=0.0):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, x):
        return self.network(x)


Create an instance of `MLP` with `hidden_dim=64` and `dropout=0.2`. Move the model to `DEVICE` and store it in the variable `model`.

In [ ]:
model = MLP(hidden_dim=64, dropout=0.2).to(DEVICE)
model

## 5. Training and Evaluation Functions

To compare hyperparameters fairly, we need reusable training and evaluation functions.


In [ ]:
def accuracy_from_logits(logits, y_true):
    predictions = torch.argmax(logits, dim=1)
    return (predictions == y_true).float().mean().item()

def train_one_epoch(model, dataloader, optimizer, loss_fn):
    model.train()
    total_loss, total_acc, n_batches = 0.0, 0.0, 0
    for X_batch, y_batch in dataloader:
        X_batch = X_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)
        optimizer.zero_grad()
        logits = model(X_batch)
        loss = loss_fn(logits, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        total_acc += accuracy_from_logits(logits, y_batch)
        n_batches += 1
    return total_loss / n_batches, total_acc / n_batches

@torch.no_grad()
def evaluate(model, dataloader, loss_fn):
    model.eval()
    total_loss, total_acc, n_batches = 0.0, 0.0, 0
    for X_batch, y_batch in dataloader:
        X_batch = X_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)
        logits = model(X_batch)
        loss = loss_fn(logits, y_batch)
        total_loss += loss.item()
        total_acc += accuracy_from_logits(logits, y_batch)
        n_batches += 1
    return total_loss / n_batches, total_acc / n_batches


## 6. Important Hyperparameters

### Learning Rate
The learning rate controls the optimizer step size. If it is too small, training can be slow. If it is too large, training can become unstable.

### Batch Size
Batch size controls how many samples are used for one gradient update. Small batches create noisier gradients; large batches create smoother gradients and often need adjusted learning rates.

### Number of Epochs
Too few epochs can cause underfitting. Too many epochs can cause overfitting.

### Weight Decay
Weight decay penalizes large weights and can improve generalization.

### Dropout
Dropout randomly disables neurons during training and can reduce overfitting.


### TODO

Complete the following configuration object by choosing reasonable starting values for `learning_rate` (usually smaller than 1), `batch_size` (something between 1 and 512), `epochs` (between 2 and 300), `weight_decay` (usually very small), `hidden_dim` (between 32 and 256), and `dropout` (between 0 and 1).

In [ ]:
# Solution
@dataclass
class TrainingConfig:
    learning_rate: float = 
    batch_size: int = 
    epochs: int =
    weight_decay: float = 
    hidden_dim: int = 
    dropout: float = 
    optimizer_name: str = "Adam"

config = TrainingConfig()
config


## 7. Train a Model with One Configuration


In [ ]:
def create_optimizer(model, config):
    if config.optimizer_name.lower() == "sgd":
        return torch.optim.SGD(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
    if config.optimizer_name.lower() == "adam":
        return torch.optim.Adam(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
    raise ValueError(f"Unknown optimizer: {config.optimizer_name}")

def run_training(config, verbose=False):
    train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=config.batch_size, shuffle=False)
    model = MLP(hidden_dim=config.hidden_dim, dropout=config.dropout).to(DEVICE)
    optimizer = create_optimizer(model, config)
    loss_fn = nn.CrossEntropyLoss()
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    for epoch in range(config.epochs):
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, loss_fn)
        val_loss, val_acc = evaluate(model, val_loader, loss_fn)
        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        if verbose and ((epoch + 1) % 10 == 0 or epoch == 0):
            print(f"Epoch {epoch + 1:03d}/{config.epochs} | train loss: {train_loss:.4f} | val loss: {val_loss:.4f} | train acc: {train_acc:.4f} | val acc: {val_acc:.4f}")
    return model, history

model, history = run_training(config, verbose=True)


Plot the training loss and validation loss over all epochs using `history["train_loss"]` and `history["val_loss"]`.

In [ ]:
plt.figure(figsize=(7, 5))
plt.plot(history["train_loss"], label="Training loss")
plt.plot(history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and validation loss")
plt.legend()
plt.show()

Plot the training accuracy and validation accuracy over all epochs using `history["train_acc"]` and `history["val_acc"]`.

In [ ]:
plt.figure(figsize=(7, 5))
plt.plot(history["train_acc"], label="Training accuracy")
plt.plot(history["val_acc"], label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training and validation accuracy")
plt.legend()
plt.show()

## 8. Hyperparameter Search

Train several models with different hyperparameter settings and choose the model with the best validation performance. The test set should only be used once at the end.


### TODO

Extend the list named `configs` with at least four different hyperparameter configurations. Vary at least learning rate, batch size, hidden dimension, and dropout.

### END TODO

In [ ]:
# TODO
configs = [
    TrainingConfig(learning_rate=0.001, batch_size=32, epochs=40, weight_decay=1e-4, hidden_dim=32, dropout=0.0, optimizer_name="Adam")
]
configs


In [ ]:
results = []
trained_models = []
for i, cfg in enumerate(configs):
    print(f"Training configuration {i + 1}/{len(configs)}: {cfg}")
    model_i, history_i = run_training(cfg, verbose=False)
    results.append({
        "config_id": i,
        "learning_rate": cfg.learning_rate,
        "batch_size": cfg.batch_size,
        "epochs": cfg.epochs,
        "weight_decay": cfg.weight_decay,
        "hidden_dim": cfg.hidden_dim,
        "dropout": cfg.dropout,
        "optimizer": cfg.optimizer_name,
        "best_val_acc": max(history_i["val_acc"]),
        "final_val_acc": history_i["val_acc"][-1],
        "final_val_loss": history_i["val_loss"][-1],
    })
    trained_models.append((model_i, history_i, cfg))
results


Find the configuration with the highest validation accuracy. Store the best result dictionary in `best_result` and the best configuration id in `best_config_id`.

In [ ]:
best_result = max(results, key=lambda item: item["best_val_acc"])
best_config_id = best_result["config_id"]
best_result

## 9. Evaluate the Best Model on the Test Set

Only after choosing the hyperparameters based on the validation set do we evaluate the selected model on the test set.


Evaluate the best model on the test set. Store the result in `test_loss` and `test_acc`.

In [ ]:
best_model, best_history, best_config = trained_models[best_config_id]
test_loader = DataLoader(test_dataset, batch_size=best_config.batch_size, shuffle=False)
loss_fn = nn.CrossEntropyLoss()
test_loss, test_acc = evaluate(best_model, test_loader, loss_fn)
print(f"Best configuration: {best_config}")
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}")

## 10. Learning Rate Experiment

The learning rate is often the most important hyperparameter. Let us compare several learning rates while keeping the other settings fixed.


### TODO

Create a list called `learning_rates` with at least four different values. Then train one model per learning rate and store the histories in `lr_histories`.

### END TODO


In [ ]:
learning_rates = []

In [ ]:
lr_histories = {}
for lr in learning_rates:
    cfg = TrainingConfig(learning_rate=lr, batch_size=32, epochs=30, weight_decay=1e-4, hidden_dim=64, dropout=0.1, optimizer_name="Adam")
    _, hist = run_training(cfg, verbose=False)
    lr_histories[lr] = hist

plt.figure(figsize=(8, 5))
for lr, hist in lr_histories.items():
    plt.plot(hist["val_loss"], label=f"lr={lr}")
plt.xlabel("Epoch")
plt.ylabel("Validation loss")
plt.title("Effect of learning rate on validation loss")
plt.legend()
plt.show()


## 11. Interpretation Questions

Answer the following questions in your own words.


Question 1: What can happen if the learning rate is too high?

**Solution:**


Question 2: Why should the test set not be used for hyperparameter selection?

**Solution:**



Question 3: What is the difference between grid search and random search?

**Solution:**


## 12. Optional: Random Hyperparameter Search

Random search can be more efficient than grid search, especially when some hyperparameters matter much more than others.


In [ ]:
def sample_random_config():
    learning_rate = 10 ** random.uniform(-4, -1)
    batch_size = random.choice([16, 32, 64, 128])
    hidden_dim = random.choice([16, 32, 64, 128])
    dropout = random.choice([0.0, 0.1, 0.2, 0.3])
    weight_decay = random.choice([0.0, 1e-5, 1e-4, 1e-3])
    optimizer_name = random.choice(["Adam", "SGD"])
    return TrainingConfig(learning_rate=learning_rate, batch_size=batch_size, epochs=25, weight_decay=weight_decay, hidden_dim=hidden_dim, dropout=dropout, optimizer_name=optimizer_name)


Run a random search with `n_trials = 5`. Store all result dictionaries in `random_search_results`.

In [ ]:
n_trials = 5
random_search_results = []
for trial in range(n_trials):
    cfg = sample_random_config()
    print(f"Trial {trial + 1}/{n_trials}: {cfg}")
    _, hist = run_training(cfg, verbose=False)
    random_search_results.append({"trial": trial, "config": cfg, "best_val_acc": max(hist["val_acc"]), "final_val_loss": hist["val_loss"][-1]})
best_random_result = max(random_search_results, key=lambda item: item["best_val_acc"])
best_random_result


## 13. PyTorch Learning Rate Schedulers

PyTorch provides several learning rate schedulers that adjust the learning rate during training. This can help improve convergence and model performance. The usage is quite easy. Just define one of the schedulers and use it in addition in your learning function (scheduler.step() adapts your learning rate based on the scheduler).

```
optimizer = optim.SGD(model.parameters(), learning_rate)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)
optimizer.zero_grad()
output = model(data)
loss = criterion(output, target)
loss.backward()
optimizer.step()
if scheduler:
    scheduler.step()
```

### Common Schedulers:

1. **StepLR**: Decays the learning rate of each parameter group by `gamma` every `step_size` epochs.
    ```python
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)
    ```

2. **MultiStepLR**: Decays the learning rate of each parameter group by `gamma` at milestones (list of epoch indices).
    ```python
    scheduler = optim.lr_scheduler.MultiStepLR(optimizer, milestones=[30, 80], gamma=0.1)
    ```

3. **ExponentialLR**: Decays the learning rate of each parameter group by `gamma` every epoch.
    ```python
    scheduler = optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.9)
    ```

4. **CosineAnnealingLR**: Sets the learning rate of each parameter group using a cosine annealing schedule.
    ```python
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)
    ```

5. **LambdaLR**: Sets the learning rate of each parameter group to the initial learning rate times a given function.
    ```python
    scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lambda epoch: 0.95 ** epoch)
    ```

6. **ReduceLROnPlateau**: Reduces the learning rate when a metric has stopped improving.
    ```python
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min')
    ```

Each scheduler can be combined with any optimizer to dynamically adjust the learning rate during training. Below, we will demonstrate different learning rate adaptation techniques using some of these schedulers.

## 14 Early Stopping and Saving Model Parameters

### Early Stopping
Early stopping is a technique used to halt the training process when the performance on a validation set starts to degrade. This helps to prevent overfitting and saves computational resources.

### Saving Model Parameters
Saving the model parameters allows you to persist the trained model for future use without needing to retrain it from scratch.


## 15. Summary

In this tutorial, you learned that hyperparameters strongly influence neural network training.

Key points:

- Parameters are learned; hyperparameters are chosen.
- Learning rate is often the most critical hyperparameter.
- Validation data is used for model selection.
- Test data should only be used for final evaluation.
- Grid search is systematic but can become expensive.
- Random search is often a strong baseline for hyperparameter optimization.
- Regularization hyperparameters such as dropout and weight decay help control overfitting.

Practical recommendation:

1. Start with a simple baseline.
2. Tune the learning rate first.
3. Use a validation set for model selection.
4. Compare multiple configurations fairly.
5. Evaluate the final selected model once on the test set.
